# LeetCode 3480 — Maximize Subarrays After Removing One Conflicting Pair bằng Python

- Link: https://leetcode.com/problems/maximize-subarrays-after-removing-one-conflicting-pair/
- Độ khó: **Hard**
- Chủ đề: **Đếm mảng con theo đầu phải**, giữ **hai giá trị lớn nhất** (top-2) của đầu trái các cặp xung đột, **gán phần lợi** cho cặp đang "chặn" duy nhất

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxSubarrays(self, n: int, conflictingPairs: List[List[int]]) -> int:
```

## 1. Mục tiêu học

1. Đếm số mảng con hợp lệ bằng cách cố định **đầu phải `r`** và tìm **đầu trái nhỏ nhất** được phép.
2. Chuẩn hóa mỗi cặp `[a, b]` thành `(min, max)`: mảng con chứa cả hai ⇔ `l ≤ min` và `max ≤ r`.
3. Thấy rằng bỏ một cặp chỉ có ích tại những `r` mà cặp đó là **cặp chặn duy nhất** (đầu trái lớn nhất, không hòa) ⇒ phần lợi `= max1 − max2`.
4. Gom phần lợi theo cặp (theo giá trị `max1`) để chọn cặp bỏ tốt nhất trong `O(n + m)`.
5. Đối chiếu với vét cạn và lời giải `O(m · (n + m))`.

## 2. Đề bài & ví dụ

Cho `n` (mảng `nums = [1, 2, …, n]`) và danh sách `conflictingPairs`, mỗi cặp `[a, b]` là hai **giá trị** xung đột. Bỏ **đúng một** cặp; sau đó đếm số mảng con khác rỗng **không chứa đồng thời** `a` và `b` của bất kỳ cặp nào còn lại. Trả về số lớn nhất có thể.

**Ví dụ 1:** `n = 4, pairs = [[2,3],[1,4]]` → `9` (bỏ `[2,3]`).

**Ví dụ 2:** `n = 5, pairs = [[1,2],[2,5],[3,5]]` → `12` (bỏ `[1,2]`).

**Ràng buộc:** `2 ≤ n ≤ 10^5`, `1 ≤ m ≤ 2n`, `a ≠ b`. Cặp có thể **trùng nhau** và có thể cho theo thứ tự `a > b`.

In [1]:
from typing import List

examples = [(4, [[2, 3], [1, 4]], 9), (5, [[1, 2], [2, 5], [3, 5]], 12)]
for n, P, exp in examples:
    print('n =', n, 'pairs =', P, '-> kỳ vọng', exp)

n = 4 pairs = [[2, 3], [1, 4]] -> kỳ vọng 9
n = 5 pairs = [[1, 2], [2, 5], [3, 5]] -> kỳ vọng 12


## 3. Kiến thức nền

### 3.1. Số mảng con

Mảng độ dài `n` có `n(n+1)/2` mảng con khác rỗng. Vì `nums[i] = i + 1`, giá trị cũng là vị trí (đánh số từ 1) ⇒ mảng con `[l..r]` chứa giá trị `x` ⇔ `l ≤ x ≤ r`.

### 3.2. Cố định đầu phải

Với mỗi `r`, các đầu trái hợp lệ tạo thành một **đoạn** `(L_r, r]` (càng sát `r` càng ít phần tử, càng dễ hợp lệ). Tổng số mảng con hợp lệ `= Σ_r (r − L_r)`.

In [2]:
def count_valid(n, pairs):
    # đếm trực tiếp theo đầu phải: L_r = max{min(a,b) : max(a,b) <= r}
    L = 0
    by_r = [[] for _ in range(n + 1)]
    for a, b in pairs:
        by_r[max(a, b)].append(min(a, b))
    total = 0
    for r in range(1, n + 1):
        for a in by_r[r]:
            L = max(L, a)
        total += r - L
    return total

print('không cặp nào:', count_valid(4, []), '= 4·5/2')
print('chỉ cặp [1,4]:', count_valid(4, [[1, 4]]))

không cặp nào: 10 = 4·5/2
chỉ cặp [1,4]: 9


### 3.3. Hai giá trị lớn nhất (top-2) cập nhật trực tuyến

Giữ `m1 ≥ m2`; thêm `x`: nếu `x > m1` thì `(m1, m2) = (x, m1)`, ngược lại nếu `x > m2` thì `m2 = x`. Chú ý khi `x == m1` thì `m2` trở thành `m1` (hai phần tử cùng lớn nhất).

In [3]:
def top2_stream(xs):
    m1 = m2 = 0
    hist = []
    for x in xs:
        if x > m1:
            m1, m2 = x, m1
        elif x > m2:
            m2 = x
        hist.append((m1, m2))
    return hist

print(top2_stream([2, 5, 3, 5, 1]))

[(2, 0), (5, 2), (5, 3), (5, 5), (5, 5)]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Chuẩn hóa cặp thành `(a, b)` với `a < b`. Cặp `(a, b)` cấm mảng con `[l..r]` khi `l ≤ a` và `b ≤ r`. Tại đầu phải `r`, gọi `m1 ≥ m2` là hai giá trị `a` lớn nhất trong các cặp có `b ≤ r`:

- Số mảng con hợp lệ kết thúc ở `r` là `r − m1`.
- Nếu bỏ cặp tạo ra `m1` (và nó là **duy nhất** đạt `m1`), số đó thành `r − m2` ⇒ lợi thêm `m1 − m2`. Bỏ cặp khác: không lợi gì ở `r`.

Tính tay ví dụ 1: cặp `(2,3)`, `(1,4)`.

| r | cặp có b ≤ r | m1, m2 | r − m1 | lợi cho cặp có a = m1 |
|---|---|---|---|---|
| 1 | – | 0, 0 | 1 | 0 |
| 2 | – | 0, 0 | 2 | 0 |
| 3 | (2,3) | 2, 0 | 1 | +2 cho a=2 |
| 4 | (2,3),(1,4) | 2, 1 | 2 | +1 cho a=2 |

Gốc = `1 + 2 + 1 + 2 = 6`, lợi lớn nhất `3` (bỏ `(2,3)`) ⇒ `9`.

In [4]:
print('gốc (không bỏ gì):', count_valid(4, [[2, 3], [1, 4]]), '| bỏ (2,3):', count_valid(4, [[1, 4]]), '| bỏ (1,4):', count_valid(4, [[2, 3]]))

gốc (không bỏ gì): 6 | bỏ (2,3): 9 | bỏ (1,4): 6


## 5. Cách 1 — Brute Force: thử bỏ từng cặp, kiểm tra từng mảng con

In [5]:
def max_sub_brute(n, pairs):
    best = 0
    for skip in range(len(pairs)):
        rest = [tuple(sorted(p)) for i, p in enumerate(pairs) if i != skip]
        c = 0
        for l in range(1, n + 1):
            for r in range(l, n + 1):
                if all(not (l <= a and b <= r) for a, b in rest):
                    c += 1
        best = max(best, c)
    return best

for n, P, exp in examples:
    print(max_sub_brute(n, P), '| kỳ vọng', exp)

9 | kỳ vọng 9
12 | kỳ vọng 12


**Độ phức tạp:** `O(m · n² · m)` — chỉ dùng làm oracle với `n ≤ 8`.

## 6. Ý tưởng thuật toán / bước đột phá

Quét `r = 1..n`, thêm các cặp có `b = r`, cập nhật `(m1, m2)`:

- `base += r − m1`
- `bonus[m1] += m1 − m2`

Đáp án `= base + max(bonus)`.

**Vì sao gán phần lợi theo giá trị `m1` là đủ?** Nếu có hai cặp khác nhau cùng đầu trái `m1` thì `m2 = m1` ⇒ phần lợi `0`. Khi chỉ một cặp đạt `m1` thì cặp đó xác định duy nhất bởi giá trị `m1` **tại thời điểm `r`** — có thể nhiều cặp có cùng `a` nhưng `b` khác nhau, khi đó cặp có `b` lớn hơn vào sau và làm `m2 = m1` ⇒ từ đó không còn lợi; các phần lợi trước đó đều thuộc về cặp vào sớm nhất với đầu trái `a`. Vì vậy `bonus[a]` luôn là phần lợi của **một** cặp cụ thể.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Thử bỏ từng cặp, mỗi lần đếm `O(n + m)`

In [6]:
def max_sub_each(n, pairs):
    best = 0
    for skip in range(len(pairs)):
        best = max(best, count_valid(n, pairs[:skip] + pairs[skip + 1:]))
    return best

for n, P, exp in examples:
    print(max_sub_each(n, P), '| kỳ vọng', exp)

9 | kỳ vọng 9
12 | kỳ vọng 12


`O(m · (n + m))` — với `n = 10^5, m = 2·10^5` là `~6·10^{10}`: quá chậm, nhưng là bước trung gian tốt để kiểm tra.

### 7.2. Cách 3 — Quét đầu phải + top-2 + bonus (lời giải chính, `O(n + m)`)

In [7]:
def max_sub_top2(n, pairs):
    by_r = [[] for _ in range(n + 1)]
    for a, b in pairs:
        if a > b:
            a, b = b, a
        by_r[b].append(a)
    m1 = m2 = 0
    base = 0
    bonus = [0] * (n + 1)
    for r in range(1, n + 1):
        for a in by_r[r]:
            if a > m1:
                m1, m2 = a, m1
            elif a > m2:
                m2 = a
        base += r - m1
        bonus[m1] += m1 - m2
    return base + max(bonus)

for n, P, exp in examples:
    print(max_sub_top2(n, P), '| kỳ vọng', exp)

9 | kỳ vọng 9
12 | kỳ vọng 12


### 7.3. Cách 4 — Góc nhìn đối xứng: quét đầu trái từ phải sang

Cố định đầu trái `l`; đầu phải phải **nhỏ hơn** `R_l = min{b : a ≥ l}`. Giữ hai giá trị `b` **nhỏ nhất** `m1 ≤ m2` (mặc định `n + 1`). `base += m1 − l`, `bonus[m1] += m2 − m1`.

In [8]:
def max_sub_left(n, pairs):
    by_l = [[] for _ in range(n + 2)]
    for a, b in pairs:
        if a > b:
            a, b = b, a
        by_l[a].append(b)
    INF = n + 1
    m1 = m2 = INF
    base = 0
    bonus = [0] * (n + 2)
    for l in range(n, 0, -1):
        for b in by_l[l]:
            if b < m1:
                m1, m2 = b, m1
            elif b < m2:
                m2 = b
        base += m1 - l
        bonus[m1] += m2 - m1
    return base + max(bonus)

for n, P, exp in examples:
    print(max_sub_left(n, P), '| kỳ vọng', exp)

9 | kỳ vọng 9
12 | kỳ vọng 12


### 7.4. Cách 5 — Theo gợi ý: `f[i]` và cập nhật tiền tố (dạng "min đoạn tiền tố")

Gợi ý của đề: `f[l]` = đầu phải xa nhất hợp lệ khi bắt đầu tại `l`, đáp án gốc `= Σ (f[l] − l + 1)`. `f[l] = min{b : a ≥ l} − 1` chính là `m1 − 1` ở cách 4 — nghĩa là cách 4 là bản "không cần segment tree" của gợi ý. Ở đây viết dạng mảng tiền tố để thấy quan hệ:

In [9]:
def f_array(n, pairs):
    best_b = [n + 1] * (n + 2)          # best_b[a] = min b của các cặp có đầu trái a
    for a, b in pairs:
        a, b = min(a, b), max(a, b)
        best_b[a] = min(best_b[a], b)
    f = [0] * (n + 2)
    cur = n + 1
    for l in range(n, 0, -1):           # min hậu tố
        cur = min(cur, best_b[l])
        f[l] = cur - 1
    return f[1:n + 1]

f = f_array(4, [[2, 3], [1, 4]])
print('f =', f, '| Σ(f[l] - l + 1) =', sum(x - l + 1 for l, x in enumerate(f, 1)), '| count_valid =', count_valid(4, [[2, 3], [1, 4]]))

f = [2, 2, 4, 4] | Σ(f[l] - l + 1) = 6 | count_valid = 6


## 8. TRACE từng bước

Trace cách 3 trên ví dụ 2.

In [10]:
def trace_top2(n, pairs):
    by_r = [[] for _ in range(n + 1)]
    for a, b in pairs:
        a, b = min(a, b), max(a, b)
        by_r[b].append(a)
    m1 = m2 = 0; base = 0; bonus = [0] * (n + 1)
    for r in range(1, n + 1):
        for a in by_r[r]:
            if a > m1: m1, m2 = a, m1
            elif a > m2: m2 = a
        base += r - m1
        bonus[m1] += m1 - m2
        print(f"r={r}: thêm a={by_r[r]}, (m1,m2)=({m1},{m2}), hợp lệ kết thúc ở r: {r - m1}, "
              f"base={base}, bonus[{m1}] += {m1 - m2}")
    print('bonus =', bonus, '-> đáp án', base, '+', max(bonus), '=', base + max(bonus))

trace_top2(*examples[1][:2])

r=1: thêm a=[], (m1,m2)=(0,0), hợp lệ kết thúc ở r: 1, base=1, bonus[0] += 0
r=2: thêm a=[1], (m1,m2)=(1,0), hợp lệ kết thúc ở r: 1, base=2, bonus[1] += 1
r=3: thêm a=[], (m1,m2)=(1,0), hợp lệ kết thúc ở r: 2, base=4, bonus[1] += 1
r=4: thêm a=[], (m1,m2)=(1,0), hợp lệ kết thúc ở r: 3, base=7, bonus[1] += 1
r=5: thêm a=[2, 3], (m1,m2)=(3,2), hợp lệ kết thúc ở r: 2, base=9, bonus[3] += 1
bonus = [0, 3, 0, 1, 0, 0] -> đáp án 9 + 3 = 12


Đọc trace: `bonus[1] = 3` (cặp `(1,2)` chặn duy nhất ở `r = 2, 3, 4`, mỗi lần lợi 1), `bonus[3] = 1` (cặp `(3,5)` chặn duy nhất ở `r = 5`) ⇒ bỏ `(1,2)` cho `9 + 3 = 12`.

## 9. Code nộp LeetCode

In [11]:
class Solution:
    def maxSubarrays(self, n: int, conflictingPairs: List[List[int]]) -> int:
        by_r = [[] for _ in range(n + 1)]
        for a, b in conflictingPairs:
            if a > b:
                a, b = b, a
            by_r[b].append(a)
        m1 = m2 = 0
        base = 0
        bonus = [0] * (n + 1)
        for r in range(1, n + 1):
            for a in by_r[r]:
                if a > m1:
                    m1, m2 = a, m1
                elif a > m2:
                    m2 = a
            base += r - m1
            bonus[m1] += m1 - m2
        return base + max(bonus)

for n, P, exp in examples:
    got = Solution().maxSubarrays(n, P)
    print(got, 'OK' if got == exp else 'SAI')

9 OK
12 OK


## 10. Vì sao thuật toán đúng

1. **Đếm gốc:** mảng con `[l..r]` hợp lệ ⇔ với mọi cặp `(a, b)` có `b ≤ r` đều có `l > a` ⇔ `l > m1(r)`. Số `l` hợp lệ là `r − m1(r)`.
2. **Bỏ cặp `c = (a_c, b_c)`:** với mỗi `r`, `m1` mới bằng `m1(r)` nếu `c` không phải cặp duy nhất đạt `m1(r)` (hoặc `b_c > r`), ngược lại bằng `m2(r)`. ⇒ số mới `= base + Σ_{r: c là cặp chặn duy nhất} (m1 − m2)`.
3. **Gán theo giá trị `m1`:** với `m1 − m2 > 0` thì có đúng một cặp đạt `m1` trong các cặp có `b ≤ r`; đó luôn là **cùng một** cặp với mọi `r` mà `m1(r) = a` và `m1 > m2` (cặp đầu tiên được thêm với đầu trái `a`; khi cặp thứ hai cùng `a` xuất hiện thì `m2 = m1`). Do đó `bonus[a]` đúng bằng phần lợi của cặp đó, và các cặp còn lại có phần lợi `0`.
4. **Đúng một cặp phải bỏ:** phần lợi luôn ≥ 0 nên `max(bonus)` (≥ 0) là lựa chọn tốt nhất; kể cả khi mọi phần lợi bằng 0 thì bỏ cặp nào cũng cho `base`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(m² n²)` | `O(m)` |
| 2. Bỏ từng cặp + đếm tuyến tính | `O(m (n + m))` | `O(n + m)` |
| 3. Top-2 theo đầu phải | `O(n + m)` | `O(n + m)` |
| 4. Top-2 theo đầu trái | `O(n + m)` | `O(n + m)` |
| 5. `f[l]` (chỉ phần đếm gốc) | `O(n + m)` | `O(n)` |

Đáp án tối đa `≈ n²/2 = 5·10^9` — Python không lo tràn số; ở C++/Java cần `long`.

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [12]:
impls = {'each': max_sub_each, 'top2': max_sub_top2, 'left': max_sub_left,
         'submit': lambda n, P: Solution().maxSubarrays(n, P)}
edge = [
    (2, [[1, 2]], 3),                    # bỏ cặp duy nhất: mọi mảng con
    (2, [[2, 1], [1, 2]], 2),            # cặp trùng (đảo thứ tự): bỏ một vẫn còn một
    (3, [[3, 1]], 6),
    (4, [[1, 2], [2, 3], [3, 4]], 5),
    (5, [[1, 5], [1, 5], [2, 4]], 14),
]
for n, P, exp in examples + edge:
    assert max_sub_brute(n, P) == exp, (n, P, max_sub_brute(n, P))
    for name, f in impls.items():
        assert f(n, P) == exp, (name, n, P)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [13]:
import random
rng = random.Random(3480)
cnt = 0
for _ in range(2500):
    n = rng.randint(2, 8)
    m = rng.randint(1, 2 * n)
    P = [rng.sample(range(1, n + 1), 2) for _ in range(m)]
    exp = max_sub_brute(n, P)
    for name, f in impls.items():
        assert f(n, P) == exp, (name, n, P)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ ngẫu nhiên (có cặp trùng, có cặp đảo thứ tự).')

Khớp vét cạn trên 2500 bộ ngẫu nhiên (có cặp trùng, có cặp đảo thứ tự).


### 12.3. Ca lớn `n = 10^5, m = 2·10^5`

In [14]:
import time
rng = random.Random(1)
n = 10 ** 5
cases = {
    'ngẫu nhiên': [rng.sample(range(1, n + 1), 2) for _ in range(2 * n)],
    'cặp kề nhau': [[i, i + 1] for i in range(1, n)] + [[i + 1, i] for i in range(1, n)],
    'một cặp lớn trùng lặp': [[1, n]] * (2 * n),
}
for name, P in cases.items():
    t = time.time(); a = Solution().maxSubarrays(n, P); t1 = time.time() - t
    t = time.time(); b = max_sub_left(n, P); t2 = time.time() - t
    assert a == b
    print(f"{name:22s}: {a} | cách 3 {t1:.2f}s | cách 4 {t2:.2f}s")

ngẫu nhiên            : 20242988 | cách 3 0.14s | cách 4 0.07s
cặp kề nhau           : 100000 | cách 3 0.03s | cách 4 0.09s


một cặp lớn trùng lặp : 5000049999 | cách 3 0.02s | cách 4 0.09s


## 13. Các lỗi người mới hay gặp

Hàm tìm phản ví dụ (seed cố định):

In [15]:
def find_counterexample(wrong, seed=0, tries=20000):
    rng = random.Random(seed)
    for _ in range(tries):
        n = rng.randint(2, 6)
        P = [rng.sample(range(1, n + 1), 2) for _ in range(rng.randint(1, 2 * n))]
        if wrong(n, P) != max_sub_brute(n, P):
            return n, P
    return None

### Lỗi 1 — Không chuẩn hóa cặp `[a, b]` khi `a > b`

In [16]:
def wrong_no_normalize(n, pairs):
    by_r = [[] for _ in range(n + 1)]
    for a, b in pairs:
        if b <= n:
            by_r[b].append(a)                # SAI: coi a là đầu trái dù a > b
    m1 = m2 = 0; base = 0; bonus = [0] * (n + 1)
    for r in range(1, n + 1):
        for a in by_r[r]:
            if a > m1: m1, m2 = a, m1
            elif a > m2: m2 = a
        base += max(0, r - m1)
        bonus[min(m1, n)] += m1 - m2
    return base + max(bonus)

print('sai :', wrong_no_normalize(4, [[3, 2], [4, 1]]), '| đúng:', Solution().maxSubarrays(4, [[3, 2], [4, 1]]))

sai : 7 | đúng: 9


**Vì sao sai:** `[3, 2]` và `[2, 3]` là cùng một ràng buộc. Đề không đảm bảo `a < b`; phải lấy `(min, max)`.

### Lỗi 2 — Cộng dồn mọi phần lợi (như thể bỏ được nhiều cặp)

In [17]:
def wrong_sum_bonus(n, pairs):
    by_r = [[] for _ in range(n + 1)]
    for a, b in pairs:
        a, b = min(a, b), max(a, b)
        by_r[b].append(a)
    m1 = m2 = 0; base = 0; gain = 0
    for r in range(1, n + 1):
        for a in by_r[r]:
            if a > m1: m1, m2 = a, m1
            elif a > m2: m2 = a
        base += r - m1
        gain += m1 - m2                      # SAI: không phân biệt cặp nào được bỏ
    return base + gain

n, P, _ = examples[1]
print('sai :', wrong_sum_bonus(n, P), '| đúng:', Solution().maxSubarrays(n, P))

sai : 13 | đúng: 12


**Vì sao sai:** ở ví dụ 2, phần lợi tại `r = 2..4` thuộc cặp `(1,2)`, tại `r = 5` thuộc cặp `(3,5)`. Chỉ được bỏ **một** cặp ⇒ phải gom theo cặp rồi lấy max.

### Lỗi 3 — Cập nhật top-2 thiếu nhánh `elif`

In [18]:
def wrong_top2(n, pairs):
    by_r = [[] for _ in range(n + 1)]
    for a, b in pairs:
        a, b = min(a, b), max(a, b)
        by_r[b].append(a)
    m1 = m2 = 0; base = 0; bonus = [0] * (n + 1)
    for r in range(1, n + 1):
        for a in by_r[r]:
            if a > m1:
                m1, m2 = a, m1               # SAI: thiếu "elif a > m2: m2 = a"
        base += r - m1
        bonus[m1] += m1 - m2
    return base + max(bonus)

ce = find_counterexample(wrong_top2, seed=3)
print('phản ví dụ:', ce)
print('sai :', wrong_top2(*ce), '| đúng:', max_sub_brute(*ce))

phản ví dụ: (3, [[3, 1], [2, 3], [3, 1], [3, 1], [2, 3]])
sai : 5 | đúng: 4


**Vì sao sai:** một cặp có đầu trái nằm giữa `m2` và `m1` (hoặc **bằng** `m1`) vẫn phải cập nhật `m2`; nếu không, phần lợi khi bỏ cặp `m1` bị **phóng đại**.

### Lỗi 4 — Gán phần lợi cho cặp theo **chỉ số cặp được thêm cuối cùng** thay vì cặp đạt `m1`

In [19]:
def wrong_last_added(n, pairs):
    by_r = [[] for _ in range(n + 1)]
    for idx, (a, b) in enumerate(pairs):
        a, b = min(a, b), max(a, b)
        by_r[b].append((a, idx))
    m1 = m2 = 0; base = 0; bonus = [0] * len(pairs); last = 0
    for r in range(1, n + 1):
        for a, idx in by_r[r]:
            last = idx                       # SAI: cặp mới thêm chưa chắc là cặp chặn
            if a > m1: m1, m2 = a, m1
            elif a > m2: m2 = a
        base += r - m1
        bonus[last] += m1 - m2
    return base + max(bonus)

ce = find_counterexample(wrong_last_added, seed=4)
print('phản ví dụ:', ce)
print('sai :', wrong_last_added(*ce), '| đúng:', max_sub_brute(*ce))

phản ví dụ: (5, [[2, 3], [4, 3], [5, 2], [3, 1], [1, 2]])
sai : 7 | đúng: 8


**Vì sao sai:** phần lợi `m1 − m2` thuộc về cặp có đầu trái `m1`, có thể đã được thêm từ rất sớm.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm mảng con không chứa cặp nào" ⇒ cố định đầu phải, đầu trái phải vượt `max a` ⇒ `Σ (r − m1)`.
- "Được bỏ đúng một ràng buộc" ⇒ chỉ ràng buộc **chặt nhất duy nhất** có ảnh hưởng ⇒ giữ **top-2**, phần lợi `m1 − m2`.
- Gom phần lợi theo "danh tính" của ràng buộc chặt nhất (ở đây chính là giá trị `m1`).
- Cẩn thận dữ liệu: cặp đảo thứ tự, cặp trùng.

## 15. Biến thể của bài

1. Không bỏ cặp nào ⇒ chỉ cần `base`.
2. Bỏ **hai** cặp ⇒ giữ top-3; phần lợi phụ thuộc cặp (top1) hoặc (top1, top2).
3. Mỗi cặp có **chi phí** bỏ; được ngân sách `C` ⇒ chọn một cặp tối đa hóa `bonus − λ·cost`…
4. Bài tương tự: LC 2444 (Count Subarrays With Fixed Bounds) — cũng đếm theo đầu phải với "vị trí gần nhất".

In [20]:
# Biến thể 2: bỏ đúng hai cặp — oracle "bỏ từng cặp" lồng nhau (dùng lại ở bài 5)
def max_sub_remove_two(n, pairs):
    best = 0
    m = len(pairs)
    for i in range(m):
        for j in range(i + 1, m):
            rest = [p for t, p in enumerate(pairs) if t not in (i, j)]
            best = max(best, count_valid(n, rest))
    return best

# Biến thể 1: không bỏ cặp nào
for n, P, _ in examples:
    print(f"n={n}, pairs={P}: không bỏ = {count_valid(n, P)}, bỏ 1 = {Solution().maxSubarrays(n, P)}, "
          f"bỏ 2 = {max_sub_remove_two(n, P)}")

n=4, pairs=[[2, 3], [1, 4]]: không bỏ = 6, bỏ 1 = 9, bỏ 2 = 10
n=5, pairs=[[1, 2], [2, 5], [3, 5]]: không bỏ = 9, bỏ 1 = 12, bỏ 2 = 13


Bỏ thêm cặp không bao giờ làm giảm số mảng con hợp lệ, nên ba con số trên không giảm. Lời giải tuyến tính cho "bỏ hai cặp" (giữ top-3) nằm ở bài tập 5.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Vét cạn | `O(m² n²)` | Oracle |
| 2 | Bỏ từng cặp + đếm tuyến tính | `O(m (n+m))` | Kiểm tra trung gian |
| 3 | Đầu phải + top-2 + bonus | `O(n + m)` | **Nộp bài** |
| 4 | Đầu trái + bottom-2 | `O(n + m)` | Đối xứng |
| 5 | `f[l]` theo gợi ý | `O(n + m)` cho đếm gốc | Liên hệ với segment tree trong gợi ý |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `n = 3, pairs = [[1, 3], [1, 3]]`.

**Bài 2 (tính tay):** `n = 5, pairs = [[2, 4], [1, 5]]`.

**Bài 3 (điền TODO):** viết `cost_of_each_pair(n, pairs)` — trả về danh sách số mảng con nhận được khi bỏ từng cặp (dùng `bonus`, `O(n + m)`).

**Bài 4 (tự viết lại):** viết lại cách 3 trả về **chỉ số** cặp nên bỏ (nhỏ nhất nếu hòa).

**Bài 5 (mini challenge):** hoàn thiện biến thể 2 (bỏ đúng hai cặp) để khớp vét cạn.

In [21]:
def cost_of_each_pair_todo(n, pairs):
    # TODO
    raise NotImplementedError

def best_pair_index_todo(n, pairs):
    # TODO
    raise NotImplementedError

def remove_two_exact_todo(n, pairs):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Hai cặp trùng. (2) Bỏ một vẫn còn một `(1,3)`. (3) `6 − 1 = 5`.

**Bài 2.** (1) Lập bảng `r, m1, m2` như mục 4. (2) `bonus[2]` tại `r = 4, 5`. (3) So với vét cạn.

**Bài 3.** (1) Cặp `c` nhận `base + bonus[a_c]` nếu nó là cặp **đầu tiên** (theo `b`, rồi theo thứ tự thêm) có đầu trái `a_c`. (2) Các cặp khác nhận `base`. (3) Đối chiếu `count_valid` bỏ từng cặp.

**Bài 4.** (1) Từ bài 3, lấy `argmax`. (2) Hòa ⇒ chỉ số nhỏ nhất. (3) Kiểm tra bằng cách 2.

**Bài 5.** (1) Với mỗi `r`, lợi khi bỏ tập `S` là `m1(r) − (max a không thuộc S)`. (2) Chỉ các tập `S` chứa top-1 có lợi; nếu `S` chứa top-1 và top-2 thì lợi `a1 − a3`, nếu chỉ chứa top-1 thì `a1 − a2`. (3) Gom `single[i]` và `double[{i,j}]`, đáp án với `{i, j}` là `double[{i,j}] + single'[i] + single'[j]` trong đó `single'` chỉ tính các `r` mà `{i, j}` không đồng thời là top-2.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2
a1 = Solution().maxSubarrays(3, [[1, 3], [1, 3]])
a2 = Solution().maxSubarrays(5, [[2, 4], [1, 5]])
assert a1 == max_sub_brute(3, [[1, 3], [1, 3]]) == 5
assert a2 == max_sub_brute(5, [[2, 4], [1, 5]])
print('Bài 1:', a1, '| Bài 2:', a2, '(bỏ (2,4):', count_valid(5, [[1, 5]]), ', bỏ (1,5):', count_valid(5, [[2, 4]]), ')')

Bài 1: 5 | Bài 2: 14 (bỏ (2,4): 14 , bỏ (1,5): 11 )


In [23]:
# Bài 3, 4
def cost_of_each_pair(n, pairs):
    by_r = [[] for _ in range(n + 1)]
    for idx, (a, b) in enumerate(pairs):
        by_r[max(a, b)].append((min(a, b), idx))
    owner = {}                                # đầu trái a -> chỉ số cặp đầu tiên có đầu trái a
    m1 = m2 = 0; base = 0; bonus = [0] * (n + 1)
    for r in range(1, n + 1):
        for a, idx in by_r[r]:
            owner.setdefault(a, idx)
            if a > m1: m1, m2 = a, m1
            elif a > m2: m2 = a
        base += r - m1
        bonus[m1] += m1 - m2
    res = [base] * len(pairs)
    for a, idx in owner.items():
        res[idx] += bonus[a]
    return res

def best_pair_index(n, pairs):
    c = cost_of_each_pair(n, pairs)
    return max(range(len(pairs)), key=lambda i: (c[i], -i))

rng = random.Random(19)
for _ in range(800):
    n = rng.randint(2, 7)
    P = [rng.sample(range(1, n + 1), 2) for _ in range(rng.randint(1, 2 * n))]
    exp = [count_valid(n, P[:i] + P[i + 1:]) for i in range(len(P))]
    assert cost_of_each_pair(n, P) == exp, (n, P)
    assert best_pair_index(n, P) == max(range(len(P)), key=lambda i: (exp[i], -i))
print('Bài 3, 4 khớp; ví dụ 2:', cost_of_each_pair(5, [[1, 2], [2, 5], [3, 5]]), '-> bỏ cặp', best_pair_index(5, [[1, 2], [2, 5], [3, 5]]))

Bài 3, 4 khớp; ví dụ 2: [12, 9, 10] -> bỏ cặp 0


In [24]:
# Bài 5 — bỏ đúng hai cặp, O((n + m) log) + số cặp ứng viên
from collections import defaultdict

def remove_two_exact(n, pairs):
    m = len(pairs)
    if m < 2:
        return count_valid(n, [])
    by_r = [[] for _ in range(n + 1)]
    for idx, (a, b) in enumerate(pairs):
        by_r[max(a, b)].append((min(a, b), idx))
    top = []
    base = 0
    single = defaultdict(int)    # lợi khi chỉ bỏ top1 (tại các r mà a1 > a2)
    pair_gain = defaultdict(int) # với {top1, top2}: (a1 - a3) - (a1 - a2) = a2 - a3 phần thêm
    for r in range(1, n + 1):
        for item in by_r[r]:
            top = sorted(top + [item], reverse=True)[:3]
        t = top + [(0, -1)] * (3 - len(top))
        (a1, i1), (a2, i2), (a3, _) = t
        base += r - a1
        if i1 != -1 and a1 > a2:
            single[i1] += a1 - a2
        if i1 != -1 and i2 != -1 and a1 > a3:
            pair_gain[frozenset((i1, i2))] += (a1 - a3) - (a1 - a2 if a1 > a2 else 0)
    best_single = sorted(single.values(), reverse=True) + [0, 0]
    best = best_single[0] + 0                  # bỏ top1 tốt nhất + một cặp vô ích (luôn có vì m >= 2)
    best = max(best, best_single[0] + best_single[1])
    for key, g in pair_gain.items():
        i, j = tuple(key)
        best = max(best, single[i] + single[j] + g)
    return base + best

rng = random.Random(55)
for _ in range(1500):
    n = rng.randint(2, 7)
    P = [rng.sample(range(1, n + 1), 2) for _ in range(rng.randint(2, 7))]
    assert remove_two_exact(n, P) == max_sub_remove_two(n, P), (n, P)
print('Bài 5 khớp vét cạn trên 1500 bộ.', remove_two_exact(5, [[1, 2], [2, 5], [3, 5]]))

Bài 5 khớp vét cạn trên 1500 bộ. 13


Giải thích bài 5: bỏ `{i, j}` thì tại mỗi `r` lợi là `a1 − a2` nếu chỉ top-1 thuộc `{i, j}`, là `a1 − a3` nếu cả top-1 và top-2 thuộc `{i, j}`, ngược lại `0`. Viết `a1 − a3 = (a1 − a2) + (a2 − a3)` ⇒ tổng lợi `= single[i] + single[j] + pair_gain[{i, j}]`. Cặp `{i, j}` không xuất hiện trong `pair_gain` thì lợi chỉ là `single[i] + single[j]` ⇒ lấy hai giá trị `single` lớn nhất.

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
by_r[max(a,b)].append(min(a,b))
m1 = m2 = 0; base = 0; bonus = [0]*(n+1)
for r in 1..n:
    for a in by_r[r]:
        if a > m1: m1, m2 = a, m1
        elif a > m2: m2 = a
    base += r - m1
    bonus[m1] += m1 - m2
return base + max(bonus)
```

**Checklist:**

- [ ] Chuẩn hóa `(min, max)` cho mỗi cặp?
- [ ] Top-2 có cả nhánh `elif a > m2` (kể cả `a == m1`)?
- [ ] Chỉ lấy **max** một phần lợi, không cộng dồn?
- [ ] Cặp trùng ⇒ phần lợi 0 (tự động vì `m2 = m1`)?
- [ ] Kết quả có thể tới `~5·10^9` (ngôn ngữ khác cần `long`)?